<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_34_llamaindex_production_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📚 **Module 34 — LlamaIndex for Production RAG (the RAG-first framework)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 34 — LlamaIndex for Production RAG

> M30 built RAG by hand. M32 wrapped it in LangChain. **LlamaIndex** is what serious teams use when RAG is the *primary* product — knowledge bases, doc-search, customer-support copilots. It has stronger document loaders, better chunking, more retrieval strategies, and a cleaner mental model **specifically for RAG**.

This module assumes you've finished **M30 (RAG)** and **M32 (LangChain)**.

### What you'll cover
1. LangChain vs LlamaIndex — when to pick which
2. The 6 core abstractions
3. Setup with free local models (no API keys)
4. **Loading documents** — directory, web, PDF
5. Building an index → query engine in 4 lines
6. **Custom chunking** — `SentenceSplitter` and the **SemanticSplitter**
7. **Re-rankers** — second-stage refinement
8. **Hierarchical retrieval** — small-chunks-for-precision + parent-context
9. Persistence — save and reload an index
10. Practice + production patterns


## 1 · LangChain vs LlamaIndex — When Each Wins

| Need | LangChain | LlamaIndex |
|---|---|---|
| General LLM pipelines (any task) | ✅ | weaker |
| Agents / tools / multi-step workflows | ✅ | weaker (use LangGraph) |
| **RAG with messy real documents** | OK | ✅ best-in-class |
| Document loaders (PDF, Notion, Slack, Confluence...) | ~80 | **~300** |
| Advanced retrievers (hybrid, hierarchical, recursive) | available | **idiomatic** |
| Stable APIs | unstable, fast-moving | more stable |
| Hosted SaaS for parsing | partner | **LlamaCloud + LlamaParse** (their own) |

**Rule of thumb:** if RAG is the headline feature, start with LlamaIndex. If RAG is one of many things, LangChain. They also compose — many production stacks use both.

## 2 · The Six Core Abstractions

| Building block | What it is |
|---|---|
| **Document** | a single source (page, file, scraped URL) with text + metadata |
| **Node** | a chunk of a Document (after splitting). What actually gets embedded |
| **Index** | a data structure over nodes (`VectorStoreIndex`, `KeywordTableIndex`, `SummaryIndex`...) |
| **Retriever** | given a query, returns relevant nodes |
| **Postprocessor** | reranks / filters retrieved nodes (cross-encoder, similarity threshold, etc.) |
| **QueryEngine** | wraps retriever + postprocessor + LLM into one `.query(...)` call |

Mental model: `Documents → [split] → Nodes → [embed + index] → Index → [retriever] → nodes → [postprocessor] → top-K → [LLM] → answer`.

In [ ]:
!pip -q install llama-index llama-index-embeddings-huggingface llama-index-llms-huggingface llama-index-postprocessor-sbert-rerank
import warnings; warnings.filterwarnings("ignore")

## 3 · Setup — Free Local Models

Same `flan-t5` + `MiniLM` stack as M32/M33 so we don't need API keys.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings

# Embed with MiniLM (small, fast, CPU-friendly)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

# LLM with flan-t5-base
Settings.llm = HuggingFaceLLM(
    tokenizer_name="google/flan-t5-base",
    model_name="google/flan-t5-base",
    context_window=512, max_new_tokens=200,
    generate_kwargs={"do_sample": False},
)

# Chunk size for the default splitter
Settings.chunk_size = 256
Settings.chunk_overlap = 32
print("LlamaIndex Settings configured (HF embed + flan-t5)")

**Key idea — `Settings` is global state.** Once you set `Settings.llm` and `Settings.embed_model`, every Index, Retriever, and QueryEngine you build picks them up automatically. For OpenAI: just swap the two `Settings` lines.

## 4 · Loading Documents

LlamaIndex has the largest document-loader ecosystem in Python. We'll show three patterns: **inline**, **directory**, **URL**.

In [ ]:
from llama_index.core import Document, SimpleDirectoryReader
import os, tempfile

# 1) Inline — for testing
inline_docs = [
    Document(text="NumPy was created by Travis Oliphant in 2006 and is the foundation of the Python scientific stack.",
             metadata={"source": "numpy", "year": 2006}),
    Document(text="PyTorch was released by Facebook AI Research in 2016 with dynamic computation graphs.",
             metadata={"source": "pytorch", "year": 2016}),
    Document(text="DeepSeek-V3 is a 671B-parameter open-weight Mixture-of-Experts model with Multi-Latent Attention.",
             metadata={"source": "deepseek", "year": 2024}),
]
print(f"inline: {len(inline_docs)} docs")

# 2) Directory — most common in production
tmp = tempfile.mkdtemp()
for d in inline_docs:
    with open(os.path.join(tmp, d.metadata["source"] + ".txt"), "w") as f:
        f.write(d.text)

dir_docs = SimpleDirectoryReader(tmp).load_data()
print(f"directory: {len(dir_docs)} docs (auto-detects .txt/.pdf/.md/.docx)")

# 3) Web — quick demo
# from llama_index.readers.web import SimpleWebPageReader
# web_docs = SimpleWebPageReader(html_to_text=True).load_data(["https://example.com"])

## 5 · Build an Index → Query Engine — 4 Lines

In [ ]:
from llama_index.core import VectorStoreIndex

index   = VectorStoreIndex.from_documents(inline_docs)
query   = index.as_query_engine(similarity_top_k=2)

response = query.query("What year was NumPy created?")
print("answer:", response)
print("sources:", [n.metadata.get("source") for n in response.source_nodes])

**Reading those 4 lines:**
1. `VectorStoreIndex.from_documents(...)` — splits, embeds, indexes (uses `Settings`)
2. `.as_query_engine(top_k=2)` — wires retriever + LLM
3. `.query(...)` — embeds question → retrieves → stuffs into prompt → answers
4. `response.source_nodes` — the chunks that grounded the answer (always available — built-in citation support)

## 6 · Custom Chunking — SentenceSplitter and SemanticSplitter

The default `SentenceSplitter` cuts at sentence boundaries near the chunk size. The `SemanticSplitterNodeParser` cuts where the embedding similarity DROPS — semantically coherent chunks.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser

# Default sentence splitter — what we used implicitly in §5
sent_splitter = SentenceSplitter(chunk_size=128, chunk_overlap=20)

long_doc = Document(text=(
    "Python's NumPy library handles numerical arrays. "
    "Pandas builds on NumPy for tabular data. "
    "Matplotlib is the foundation for visualisation. "
    "Switching topic. Football is a popular sport. "
    "The 2022 World Cup was won by Argentina. "
    "Switching again. PyTorch is a deep learning framework. "
    "It uses dynamic computation graphs. "
))

sent_nodes = sent_splitter.get_nodes_from_documents([long_doc])
print(f"SentenceSplitter produced {len(sent_nodes)} nodes:")
for i, n in enumerate(sent_nodes):
    print(f"  [{i}] {n.text[:80]}…")

# Semantic splitter — splits when embeddings DIVERGE
sem_splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=85,
    embed_model=Settings.embed_model,
)
sem_nodes = sem_splitter.get_nodes_from_documents([long_doc])
print(f"\nSemanticSplitter produced {len(sem_nodes)} nodes:")
for i, n in enumerate(sem_nodes):
    print(f"  [{i}] {n.text[:120]}…")

**Compare the splits.** SentenceSplitter cuts every ~128 chars regardless of topic. SemanticSplitter ideally puts NumPy/Pandas together, football together, PyTorch together — three coherent chunks.

## 7 · Re-Rankers — Second-Stage Refinement

Same idea as M30 §7 (cross-encoder reranking). LlamaIndex makes it a one-line postprocessor.

In [ ]:
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=2,                       # keep the best 2 after rerank
)

query = index.as_query_engine(
    similarity_top_k=5,             # retrieve 5 cheap candidates
    node_postprocessors=[reranker], # then rerank to keep the top 2
)

response = query.query("What does DeepSeek-V3 use for attention?")
print("answer:", response)
print("sources:", [n.metadata.get("source") for n in response.source_nodes])

**Two-stage retrieval pattern:** retrieve 5 cheap candidates with a vector index, then a cross-encoder reranks to keep the most relevant 2. Production default.

## 8 · Hierarchical Retrieval — Small Chunks + Parent Context

The problem: small chunks → precise retrieval but lose context. Big chunks → richer context but noisier retrieval. Solution: **embed small chunks, but return their parent paragraph for the LLM to read.**

In [ ]:
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core import StorageContext
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# Build a hierarchy: top-level (1024) → mid (256) → leaf (64) chars
hier_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[1024, 256, 64])
hier_nodes  = hier_parser.get_nodes_from_documents(inline_docs)
leaf_nodes  = get_leaf_nodes(hier_nodes)

# Index ONLY the leaf nodes — small + precise
storage = StorageContext.from_defaults()
storage.docstore.add_documents(hier_nodes)              # all levels in docstore
leaf_index = VectorStoreIndex(leaf_nodes, storage_context=storage)

# AutoMergingRetriever: when many sibling leaves match, return the PARENT
base_retriever  = leaf_index.as_retriever(similarity_top_k=3)
auto_retriever  = AutoMergingRetriever(base_retriever, storage)

query = RetrieverQueryEngine.from_args(auto_retriever)
print(query.query("What is DeepSeek-V3?"))

**Why this matters:** when 2+ leaf chunks of the same parent paragraph all hit a query, the retriever upgrades to returning the parent — giving the LLM the FULL paragraph context instead of fragments.

## 9 · Persistence — Save / Reload the Index

Re-embedding is expensive. In production you build the index once, persist it, and reload on every server start.

In [ ]:
import tempfile
persist_dir = tempfile.mkdtemp(prefix="llamaindex_")

# Save
index.storage_context.persist(persist_dir=persist_dir)
print("saved to:", persist_dir)
print("contents:", os.listdir(persist_dir))

# Reload (in a fresh process / pod / restart)
from llama_index.core import StorageContext, load_index_from_storage

storage = StorageContext.from_defaults(persist_dir=persist_dir)
reloaded = load_index_from_storage(storage)

resp = reloaded.as_query_engine().query("Who created NumPy?")
print("\nafter reload:", resp)

For production: swap the default file-based store for **Chroma**, **Pinecone**, **Weaviate**, **Qdrant**, or **Postgres+pgvector** (M36). The graph code doesn't change — just the `vector_store` argument to `VectorStoreIndex`.

## 10 · Where This Scales

| Need | Tool |
|---|---|
| **Messy PDFs / tables / images** | **LlamaParse** (paid SaaS by LlamaIndex; first 1k pages/day free) |
| Hosted RAG-as-a-Service | **LlamaCloud** |
| Better retrievers | **Hybrid** (vector + BM25), **Recursive**, **Auto-Merging**, **HyDE** |
| Eval framework | **Trulens**, **Ragas** (M31) |
| Production vector backends | Chroma / Pinecone / Weaviate / Qdrant (M42) |

### Production patterns to know
- **Metadata filtering** — restrict retrieval by date, owner, tenant: `query.retrieve(query, filters=MetadataFilters([ExactMatchFilter(key="tenant", value="acme")]))`. Critical for multi-tenant apps.
- **Composite retrievers** — combine vector + keyword + reranker. `QueryFusionRetriever` does this.
- **Citation-mode answers** — `CitationQueryEngine` wraps the response with `[1][2]` source markers automatically.
- **Streaming** — `query.query(..., streaming=True)` returns token-by-token like ChatGPT.
- **Batch ingestion** — `IngestionPipeline` for big corpora with caching, dedup, transformations.

## Practice
1. **Web RAG** — load a Wikipedia article via `SimpleWebPageReader`, build the index, ask 3 questions about its content.
2. **Hybrid retriever** — combine `VectorIndexRetriever` + `BM25Retriever` with `QueryFusionRetriever`. Compare answers.
3. **Metadata filter** — add a `year` field to each Document, then query restricted to `year >= 2020`.
4. **Citation engine** — replace `as_query_engine` with `CitationQueryEngine.from_args(index)` and observe the answer's `[1]`-style markers.

## Recap

✅ Pick LangChain vs LlamaIndex correctly (RAG-first → LlamaIndex)
✅ Use the 6 core abstractions: Document, Node, Index, Retriever, Postprocessor, QueryEngine
✅ Build a working query engine in 4 lines of code
✅ Choose the right chunking strategy — Sentence vs Semantic vs Hierarchical
✅ Add a cross-encoder reranker as a one-line postprocessor
✅ Set up auto-merging hierarchical retrieval for small-chunk precision + big-chunk context
✅ Persist and reload the index across process boundaries

### Next module
**Module 35 — DSPy: Programmatic Prompts** — instead of hand-tuning prompt strings, declare what your model should DO and let DSPy auto-tune the prompts.
